# TP 3.2 - Agent MCP (remote + local)

Dans cette version, l'agent consomme les outils via MCP uniquement :
- DeepWiki MCP (remote)
- Tavily MCP (remote)
- Google Maps via OpenMCP (remote)
- Serveur MCP local (une seule tool : `rag_search`)

In [ ]:
import sys
from urllib.parse import urlencode

from pydantic_ai import Agent
from pydantic_ai.mcp import MCPServerStdio, MCPServerStreamableHTTP
from pydantic_ai.models.google import GoogleModel
from pydantic_ai.providers.google import GoogleProvider

from shared.agent_utils import run_agent_realtime_logging
from shared.config import ROOT_DIR, google_model_settings, project_settings

---
## 0. Configuration partagée

In [ ]:
model = ...
settings = ...

In [ ]:
system_prompt = """

# RÈGLES

Tu es un assistant de planification de voyage orienté RAG.
Tu dois produire des recommandations personnalisées à partir de faits récupérés par outils.

## Règles générales

### Usage des outils et du contexte
- Utiliser les outils avant de répondre dès qu'un fait est nécessaire.
- Chaîner les outils si besoin (date -> geocode -> météo, docs -> web_search -> web_extract, etc.).
- N'utiliser que des faits issus des outils et du contexte récupéré.
- Citer les preuves factuelles (ex: [tool_retrieve_docs:source], [web_search:url]).
- Si les données sont partielles ou contradictoires, le dire explicitement.
- Si une information manque, la lister clairement sans inventer.
- Ignorer le contexte hors sujet.

### Style de réponse
- Réponse concise et précise.
- Pas de Markdown décoratif.
- Pour chaque recommandation: 1 raison + 1 détail pratique (horaire, lieu, budget, logistique).
- Préférer des actions concrètes aux formulations vagues.

### Format attendu
1) Résumé: réponse directe
2) Plan: séquence concrète (jour par jour ou par objectif)
3) Budget: estimation par catégorie uniquement sur faits disponibles
4) Preuves: faits clés utilisés (sources/outils)
5) Informations manquantes: liste courte et explicite

Important: ne pas interrompre avec des questions en cours de route. Produire la meilleure réponse possible avec les données disponibles, puis lister les manques.

## Consignes spécifiques par outil

"""

### Déclaration des serveurs MCP

Remarques :
- Tavily utilise la clé `TAVILY_API_KEY`
- Google Maps via OpenMCP utilise la clé `GOOGLE_API_KEY`
- Comme plusieurs serveurs MCP, DeepWiki est accessible directement sans clé d'API !
- Le serveur local expose uniquement des outils complémentaires ou qui nécéssitent des données locales, comme `rag_search` (défini dans le notebook 3_1).

    --> On n'a pas besoin de lancer le serveur manuellement le MCP local grace a MCPServerStdio,
        il sera lancé automatiquement à la première requete.

#### Config des serveurs MCP


In [ ]:
tavily_query = urlencode({"tavilyApiKey": project_settings.tavily_api_key})
tavily_url = f"https://mcp.tavily.com/mcp/?{tavily_query}"

google_maps_url = "https://mcp.open-mcp.org/api/server/google-maps@latest/mcp"
google_maps_headers = {"FORWARD_VAR_KEY": project_settings.google_api_key}

# TODO : Completer les paramètres pour chaque serveur MCP

local_mcp_server = MCPServerStdio(
    command=sys.executable,
    args=..., # Indiquez le chemin du serveur local
    cwd=str(ROOT_DIR),
    tool_prefix="local", # Nom d'usage
    timeout=20, # En secondes
)

deepwiki_mcp_server = MCPServerStreamableHTTP(
    url=...,
    tool_prefix="deepwiki",
)

tavily_mcp_server = MCPServerStreamableHTTP(
    url=...,
    tool_prefix="tavily",
)

google_maps_mcp_server = MCPServerStreamableHTTP(
    url=...,
    headers=...,
    tool_prefix="gmaps",
)

#### Lancement direct des serveurs MCP

In [ ]:
mcp_toolsets = [...] # TODO : Lister les serveurs MCP à utiliser

for server in mcp_toolsets:
    server_name = server.tool_prefix or server.__class__.__name__
    try:
        tools = await server.list_tools()
        print(f"[OK] {server_name} : {len(tools)} tools")
    except Exception as error:
        raise ValueError(
            f"Serveur MCP indisponible : {server_name} ({type(error).__name__} - {error})"
        ) from error

agent_mcp = Agent(
    model=model,
    instructions=system_prompt,
    model_settings=settings,
    toolsets=mcp_toolsets,
)

print(f"Serveurs MCP configurés : {len(mcp_toolsets)}")

Résultat attendu :

```
[OK] local : 1 tools
[OK] deepwiki : 3 tools
[OK] tavily : 5 tools
[OK] gmaps : 18 tools
Serveurs MCP configurés : 4
```

Vous pouvez regarder la doc pour savoir quels sont les outils disponibles sur chaque serveur MCP.

---
### 1. Use case 1 : planning voyage

In [ ]:
prompt_use_case_1 = (
    "Je vais à Rome la semaine prochaine pour 4 jours (du jeudi au dimanche), "
    "arrivée le matin, départ le soir. "
    "Fais un plan de 4 jours avec un budget de 300 EUR pour les sorties et restaurants."
)

result_use_case_1 = await run_agent_realtime_logging(
    agent=...,
    prompt=...,
    log_path=ROOT_DIR / "TP3_travel_planner_Agent" / "logs" / "trace_3_2_use_case_1.log",
    max_steps=12,
)

In [ ]:
print(result_use_case_1.output)

---
### 2. Use case 2 : meilleure période Paris -> New York

In [ ]:
prompt_use_case_2 = (
    "Trouve la meilleure période dans les 6 prochains mois pour un voyage Paris -> New York. "
    "Compare la météo et les prix saisonniers, puis recommande une période avec justification."
)

result_use_case_2 = await run_agent_realtime_logging(
    agent=...,
    prompt=...,
    log_path=ROOT_DIR / "TP3_travel_planner_Agent" / "logs" / "trace_3_2_use_case_2.log",
    max_steps=12,
)

In [ ]:
print(result_use_case_2.output)

---
### 3. Use case 3 : recommandations près d'une adresse

In [ ]:
prompt_use_case_3 = (
    "Recommande des restaurants et des activités près de cette adresse : "
    "10 Rue de la Paix, 75002 Paris, France. "
    "Je veux des options variées et un budget modéré."
)

result_use_case_3 = await run_agent_realtime_logging(
    agent=...,
    prompt=...,
    log_path=ROOT_DIR / "TP3_travel_planner_Agent" / "logs" / "trace_3_2_use_case_3.log",
    max_steps=12,
)

In [ ]:
print(result_use_case_3.output)